# Build a Multilingual Phone Voice Agent with Bolna and Sarvam AI

This guide builds a phone-based voice agent on [Bolna](https://www.bolna.ai/) with Sarvam AI for both sides of the speech pipeline:

- **Speech to text:** `saaras:v3`
- **Text to speech:** `bulbul:v3`
- **Languages in this example:** English (`en`) and Hindi (`hi`), with mid-call switching

The notebook uses Bolna's hosted API so the result can place real telephone calls. The same configuration can also be created in the Bolna dashboard.


## Architecture

```mermaid
flowchart LR
    caller([Caller]) <--> phone[Telephony]
    phone <--> bolna[Bolna voice agent]
    bolna --> stt[Sarvam Saaras v3 STT]
    stt --> llm[LLM and agent logic]
    llm --> tts[Sarvam Bulbul v3 TTS]
    tts --> bolna
```

Bolna manages telephony, turn-taking, the LLM, language switching, and call records. Sarvam converts the caller's audio to text and the agent's response back to audio.


## Prerequisites

- A [Sarvam AI account and API key](https://dashboard.sarvam.ai/)
- A [Bolna account](https://platform.bolna.ai/) and a [Bolna API key](https://www.bolna.ai/docs/api-reference/authentication)
- Python 3.9+
- For the optional live-call step, a recipient number in E.164 format (for example, `+919876543210`) and sufficient Bolna wallet balance

> Phone calls incur usage charges. The final call cell is disabled by default.


## 1. Install the small client dependencies


In [ ]:
%pip install -q "python-dotenv>=1.0.1" "requests>=2.32.3"

## 2. Configure credentials safely

Create a local `.env` file next to this notebook. The repository already ignores `integrations/.env`. Never paste a real key into the notebook or commit `.env`.

```dotenv
BOLNA_API_KEY=your-bolna-api-key
SARVAM_API_KEY=your-sarvam-api-key
RECIPIENT_PHONE_NUMBER=+919876543210
```

Bolna uses bearer authentication for its API. Sarvam credentials can be connected from [Bolna's Sarvam integration page](https://platform.bolna.ai/auth/sarvam), or added through the documented provider API in the next cell.


In [ ]:
import os

import requests
from dotenv import load_dotenv

load_dotenv()

BOLNA_API_KEY = os.environ["BOLNA_API_KEY"]
SARVAM_API_KEY = os.environ["SARVAM_API_KEY"]
BOLNA_BASE_URL = "https://api.bolna.ai"
HEADERS = {
    "Authorization": f"Bearer {BOLNA_API_KEY}",
    "Content-Type": "application/json",
}

# A read-only request verifies the Bolna key before any configuration is changed.
me = requests.get(f"{BOLNA_BASE_URL}/user/me", headers=HEADERS, timeout=30)
me.raise_for_status()
print("Bolna API key verified.")

### Connect the Sarvam credential

Run this once per Bolna account. `provider_name="SARVAM"` is Bolna's provider key for a Sarvam API credential. If the provider is already connected, leave `ADD_SARVAM_PROVIDER` as `False`.


In [ ]:
ADD_SARVAM_PROVIDER = False

if ADD_SARVAM_PROVIDER:
    response = requests.post(
        f"{BOLNA_BASE_URL}/providers",
        headers=HEADERS,
        json={"provider_name": "SARVAM", "provider_value": SARVAM_API_KEY},
        timeout=30,
    )
    response.raise_for_status()
    print(response.json())
else:
    print("Skipped. Set ADD_SARVAM_PROVIDER=True only if Sarvam is not connected yet.")

## 3. Define the multilingual agent

The base audio configuration starts in English. `multilingual_config` adds Hindi and allows both an explicit caller request and automatic detection to trigger a switch. Bolna resolves the short language keys (`en`, `hi`) to each provider's required format; Sarvam receives BCP-47 codes such as `en-IN` and `hi-IN`.

The LLM block follows Bolna's current API quickstart. Sarvam is used for STT and TTS; you can replace the LLM with any model enabled for your Bolna account.


In [ ]:
english_voice = {
    "provider": "sarvam",
    "provider_config": {
        "voice": "shubh",
        "voice_id": "shubh",
        "model": "bulbul:v3",
        "language": "en-IN",
        "speed": 1.0,
    },
    "stream": True,
    "buffer_size": 200,
    "audio_format": "wav",
}

hindi_voice = {
    "provider": "sarvam",
    "provider_config": {
        "voice": "ritu",
        "voice_id": "ritu",
        "model": "bulbul:v3",
        "language": "hi-IN",
        "speed": 1.0,
    },
    "stream": True,
    "buffer_size": 200,
    "audio_format": "wav",
}

agent_payload = {
    "agent_config": {
        "agent_name": "Sarvam Multilingual Support Agent",
        "agent_welcome_message": "Hello! I can help in English or Hindi. How may I help you?",
        "tasks": [
            {
                "task_type": "conversation",
                "toolchain": {
                    "execution": "sequential",
                    "pipelines": [["transcriber", "llm", "synthesizer"]],
                },
                "tools_config": {
                    "llm_agent": {
                        "agent_type": "simple_llm_agent",
                        "agent_flow_type": "streaming",
                        "llm_config": {
                            "provider": "openai",
                            "model": "gpt-4.1-mini",
                            "max_tokens": 150,
                            "temperature": 0.2,
                        },
                    },
                    "transcriber": {
                        "provider": "sarvam",
                        "model": "saaras:v3",
                        "language": "en-IN",
                        "stream": True,
                        "encoding": "linear16",
                        "sampling_rate": 16000,
                    },
                    "synthesizer": english_voice,
                    "input": {"provider": "plivo", "format": "wav"},
                    "output": {"provider": "plivo", "format": "wav"},
                    "multilingual_config": {
                        "enabled": True,
                        "active_language": "en",
                        "language_switch_trigger": "requested_or_auto_detected",
                        "languages": {
                            "en": {
                                "transcriber": {"language": "en"},
                                "synthesizer": english_voice,
                                "system_prompt": (
                                    "You are a helpful customer-support agent. "
                                    "Reply briefly and clearly in English."
                                ),
                                "agent_name": "Asha",
                            },
                            "hi": {
                                "transcriber": {"language": "hi"},
                                "synthesizer": hindi_voice,
                                "system_prompt": (
                                    "आप एक सहायक ग्राहक सेवा एजेंट हैं। "
                                    "हिंदी में संक्षिप्त और स्पष्ट उत्तर दें।"
                                ),
                                "handoff_message": "ठीक है, अब मैं हिंदी में बात करूँगी।",
                                "agent_name": "आशा",
                            },
                        },
                    },
                },
                "task_config": {
                    "call_terminate": 120,
                    "hangup_after_silence": 15,
                },
            }
        ],
    },
    "agent_prompts": {
        "task_1": {
            "system_prompt": (
                "You are a helpful customer-support agent. "
                "Keep every reply under two sentences."
            )
        }
    },
}

## 4. Create the Bolna agent

`POST /v2/agent` returns the new `agent_id`. Creating an agent changes your Bolna account, so this cell is gated. Review the payload first, then enable it.


In [ ]:
CREATE_AGENT = False
agent_id = None

if CREATE_AGENT:
    response = requests.post(
        f"{BOLNA_BASE_URL}/v2/agent",
        headers=HEADERS,
        json=agent_payload,
        timeout=30,
    )
    response.raise_for_status()
    agent_id = response.json()["agent_id"]
    print("Created agent:", agent_id)
else:
    print("Payload is ready. Set CREATE_AGENT=True after reviewing it.")

## 5. Place a test call

Bolna's call API requires an `agent_id` and a recipient number in E.164 format. If you created the agent in an earlier session, paste its ID into `EXISTING_AGENT_ID`. Leave `PLACE_CALL=False` until you are ready to incur call usage.


In [ ]:
PLACE_CALL = False
EXISTING_AGENT_ID = ""
recipient = os.getenv("RECIPIENT_PHONE_NUMBER", "")
selected_agent_id = agent_id or EXISTING_AGENT_ID

if PLACE_CALL:
    if not selected_agent_id:
        raise ValueError("Set EXISTING_AGENT_ID or run the create-agent cell first.")
    if not recipient.startswith("+"):
        raise ValueError("RECIPIENT_PHONE_NUMBER must be in E.164 format.")
    response = requests.post(
        f"{BOLNA_BASE_URL}/call",
        headers=HEADERS,
        json={
            "agent_id": selected_agent_id,
            "recipient_phone_number": recipient,
        },
        timeout=30,
    )
    response.raise_for_status()
    print(response.json())
else:
    print("Call not placed. Set PLACE_CALL=True when you are ready.")

## Dashboard equivalent

If you prefer the UI:

1. Connect the Sarvam key from Bolna's [Sarvam integration page](https://platform.bolna.ai/auth/sarvam).
2. Create or open an agent. In the **Agent** tab, add English and Hindi, choose the default language, and write a native-script prompt for each language.
3. In the **Languages** tab, configure each language separately. Choose **Sarvam** for transcription and voice, then select `saaras:v3`, `bulbul:v3`, and an available Bulbul v3 voice.
4. Set language switching to **requested or auto detected** (or choose caller-requested switching if that better fits the use case).
5. Use **Speak to your agent** to place a test call.

Bolna's dashboard may evolve; the linked setup pages are the source of truth for the current labels and available voices.


## Test the multilingual behavior

During a test call, verify these cases:

1. Speak a complete English sentence; the agent should answer in English.
2. Ask, `Can you speak in Hindi?`; the agent should play the Hindi handoff message and continue in Hindi.
3. Speak a complete Hindi sentence; with automatic switching enabled, the active recognizer, prompt, and voice should switch together.
4. Mix one English word into a Hindi sentence; ordinary code-mixing should not cause repeated switching.
5. Check the execution in Bolna's call history for the transcript, status, latency, and recording.

For production, test with real accents, code-mixed speech, noisy lines, and the telephony provider you plan to deploy.


## Troubleshooting

- **401 from Bolna:** verify `BOLNA_API_KEY` and the `Bearer` prefix.
- **Sarvam authentication error:** reconnect the key and make sure it has not expired.
- **Agent does not switch languages:** confirm both languages are present, each has a voice, and the switch trigger is not restricted to caller requests.
- **Wrong or garbled transcription:** use an explicit language when it is known, and verify the actual call audio path and sample rate.
- **Call does not start:** check wallet balance, the E.164 recipient number, and the selected outbound phone number or telephony provider.


## References

- [Sarvam Speech-to-Text overview](https://docs.sarvam.ai/api/api-guides-tutorials/speech-to-text/overview)
- [Sarvam streaming Speech-to-Text API](https://docs.sarvam.ai/api/api-guides-tutorials/speech-to-text/streaming-api)
- [Sarvam Text-to-Speech voices](https://docs.sarvam.ai/api/api-guides-tutorials/text-to-speech/how-to/change-the-speaker-voice)
- [Bolna Sarvam transcriber](https://www.bolna.ai/docs/sarvam-transcriber)
- [Bolna Sarvam voice](https://www.bolna.ai/docs/sarvam-voice)
- [Bolna multilingual configuration reference](https://www.bolna.ai/docs/customizations/multilingual-config-reference)
- [Bolna Languages tab](https://www.bolna.ai/docs/agent-setup/audio-tab)
- [Bolna API quickstart](https://www.bolna.ai/docs/quickstarts/api)
- [Bolna provider API](https://www.bolna.ai/docs/api-reference/providers/add)
